# US Hospital Jobs — Data Analysis Project

A beginner-friendly, end-to-end analysis of US hospital job postings. Explore the raw data, document data quality, clean and engineer useful fields, visualize hiring patterns, review salary coverage, and export a cleaned CSV and dashboard data.

**Dataset:** `us-hospital-jobs-sample.csv`  
**Unit of analysis:** one job-posting record. Counts describe postings in this sample, not the total number of vacancies or hires.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_FILE = Path("us-hospital-jobs-sample.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Working folder: {Path.cwd()}")
print(f"Input file exists: {DATA_FILE.exists()}")

## 1. Load and understand the data

Start by checking the file, its dimensions, column names, data types, and a few example rows. Keep the original data in `df_raw` so we can compare it with the cleaned version.

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Place the CSV in the same folder as this notebook."
    )

df_raw = pd.read_csv(DATA_FILE)
print(f"Rows: {len(df_raw):,} | Columns: {df_raw.shape[1]}")
display(df_raw.head())
display(pd.DataFrame({
    "column": df_raw.columns,
    "data_type": df_raw.dtypes.astype(str).values,
    "missing_rows": df_raw.isna().sum().values,
    "missing_percent": (df_raw.isna().mean().values * 100).round(1),
}))

## 2. Check data quality

We count exact duplicates, repeated application links, missing values, and invalid posting dates. Missing salary and department values are useful findings, so they will be reported rather than filled with guesses.

In [ ]:
print(f"Exact duplicate rows: {df_raw.duplicated().sum():,}")
if "url" in df_raw.columns:
    print(f"Repeated application URLs: {df_raw.duplicated(subset=['url']).sum():,}")

missing_report = (
    df_raw.isna().sum()
    .rename("missing_rows")
    .to_frame()
    .assign(missing_percent=lambda table: (table["missing_rows"] / len(df_raw) * 100).round(1))
    .sort_values("missing_percent", ascending=False)
)
display(missing_report)

if "posted_at" in df_raw.columns:
    parsed_dates = pd.to_datetime(df_raw["posted_at"], errors="coerce")
    print(f"Dates that cannot be parsed: {parsed_dates.isna().sum():,}")
    print(f"Date range: {parsed_dates.min()} to {parsed_dates.max()}")

## 3. Clean and prepare

The cleaning steps are intentionally conservative:

- Trim extra whitespace from text and standardize source-platform names.
- Convert dates and salary values into analysis-friendly types.
- Remove duplicate application URLs, if any exist.
- Keep missing information missing; add clear labels only in analysis-friendly category columns.
- Mark invalid salary ranges rather than using them in salary calculations.

In [ ]:
df = df_raw.copy()

# Trim whitespace without changing meaningful capitalization in titles or names.
text_columns = df.select_dtypes(include=["object", "string"]).columns
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()
    df[column] = df[column].replace("", pd.NA)

# Dates and salaries should have numeric/date types for analysis.
df["posted_at"] = pd.to_datetime(df["posted_at"], errors="coerce")
for column in ["salary_min", "salary_max", "system_beds"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

if "source" in df.columns:
    df["source"] = df["source"].str.lower()
if "system_state" in df.columns:
    df["system_state"] = df["system_state"].str.upper()

rows_before_deduplication = len(df)
if "url" in df.columns:
    df = df.drop_duplicates(subset=["url"], keep="first").copy()
print(f"Duplicate URL rows removed: {rows_before_deduplication - len(df):,}")

# Standardize common variations while preserving uncommon categories.
employment_labels = {
    "full time": "Full time",
    "part time": "Part time",
    "prn": "PRN",
    "per diem": "Per diem",
    "contract": "Contract",
    "temporary": "Temporary",
    "intern": "Intern",
    "variable time": "Variable time",
    "flexible time": "Flexible time",
}
employment_source = df["employment_type"].str.lower() if "employment_type" in df.columns else pd.Series(pd.NA, index=df.index, dtype="string")
df["employment_group"] = employment_source.map(employment_labels).fillna("Not specified")

# A valid salary range must have non-negative bounds and a maximum >= minimum.
salary_bounds_are_valid = (
    df["salary_min"].notna()
    & df["salary_max"].notna()
    & df["salary_min"].ge(0)
    & df["salary_max"].ge(df["salary_min"])
)
df["salary_reported"] = salary_bounds_are_valid
df["salary_midpoint"] = ((df["salary_min"] + df["salary_max"]) / 2).where(salary_bounds_are_valid)
print(f"Rows with usable salary ranges: {df['salary_reported'].sum():,}")
print(f"Rows with incomplete or invalid salary ranges: {(~df['salary_reported']).sum():,}")

display(df.head())

## 4. Engineer useful features

These features make common questions easier to answer. The job-family labels are simple keyword-based groupings of titles, not official occupation codes. The system bed groups describe the system-level bed field, not the size of an individual facility.

In [ ]:
def classify_job_family(title):
    """Assign a broad, explainable category based on keywords in a job title."""
    title = str(title).lower()
    if re.search(r"physician|surgeon|doctor|md\b|do\b", title):
        return "Physician"
    if re.search(r"\brn\b|\bnurse\b|nursing|\blpn\b|\blvn\b", title):
        return "Nursing"
    if re.search(r"patient access|registration|scheduler|medical receptionist", title):
        return "Patient access / administration"
    if re.search(r"technologist|technician|therapist|radiolog|laboratory|\blab\b|imaging|respiratory", title):
        return "Allied health / technical"
    if re.search(r"\baide\b|assistant|support|housekeeper|food service|transport", title):
        return "Support services"
    return "Other / not specified"


df["job_family"] = df["title"].map(classify_job_family)
df["title_length"] = df["title"].fillna("").str.len()
df["title_word_count"] = df["title"].fillna("").str.split().str.len()
df["posted_year"] = df["posted_at"].dt.year
df["posted_month"] = df["posted_at"].dt.to_period("M").astype("string")
df["posted_weekday"] = df["posted_at"].dt.day_name()
df["department_clean"] = df["department"].fillna("Not specified")

df["system_bed_size_group"] = pd.cut(
    df["system_beds"],
    bins=[0, 1000, 3000, 6000, np.inf],
    labels=["Up to 1,000", "1,001–3,000", "3,001–6,000", "Over 6,000"],
    include_lowest=True,
)

display(df[[
    "title", "job_family", "employment_type", "employment_group",
    "posted_at", "posted_weekday", "system_beds", "system_bed_size_group",
    "salary_min", "salary_max", "salary_midpoint", "salary_reported",
]].head(10))

## 5. Explore hiring patterns

Compare posting counts across systems, system states, job families, employment labels, and job-board platforms. These are counts of records in the supplied snapshot.

In [ ]:
print(f"Cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Health systems: {df['health_system'].nunique():,}")
print(f"System states: {df['system_state'].nunique():,}")
print(f"Posting dates: {df['posted_at'].min().date()} through {df['posted_at'].max().date()}")

for column, label in [
    ("system_state", "Top health-system states"),
    ("health_system", "Top health systems"),
    ("job_family", "Job families"),
    ("employment_group", "Employment type"),
    ("source", "Posting sources"),
]:
    print(f"\n{label}")
    display(df[column].value_counts(dropna=False).head(10).rename("postings").to_frame())

## 6. Visualize the sample

The charts below use Matplotlib and are also saved in `outputs/` for use in a portfolio or report.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
chart_items = [
    ("system_state", "Top 10 health-system states"),
    ("job_family", "Postings by job family"),
    ("source", "Postings by source platform"),
    ("employment_group", "Postings by employment type"),
]

for axis, (column, title) in zip(axes.flat, chart_items):
    counts = df[column].fillna("Not specified").value_counts().head(10).sort_values()
    counts.plot(kind="barh", ax=axis, color="#4263eb")
    axis.set_title(title, loc="left", fontweight="bold")
    axis.set_xlabel("Number of postings")
    axis.set_ylabel("")

fig.suptitle("US Hospital Job Postings — Sample Overview", fontsize=17, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "posting_overview.png", dpi=160, bbox_inches="tight")
plt.show()

daily_postings = df.dropna(subset=["posted_at"]).groupby("posted_at").size()
fig, axis = plt.subplots(figsize=(12, 4))
daily_postings.plot(kind="bar", ax=axis, color="#0f9f92")
axis.set_title("Postings by date", loc="left", fontweight="bold")
axis.set_xlabel("Posting date")
axis.set_ylabel("Number of postings")
axis.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "postings_by_date.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Review salary coverage

Most source records do not include salary values. The midpoint is calculated only when both bounds are present and valid. Since pay periods and currencies may differ, this section reports coverage and groups available records by their stated currency and pay period rather than presenting a misleading combined average.

In [ ]:
salary_coverage = pd.DataFrame({
    "status": ["Usable salary range", "No usable salary range"],
    "postings": [
        int(df["salary_reported"].sum()),
        int((~df["salary_reported"]).sum()),
    ],
})
salary_coverage["percent"] = (salary_coverage["postings"] / len(df) * 100).round(1)
display(salary_coverage)

salary_rows = df.loc[df["salary_reported"]].copy()
if salary_rows.empty:
    print("No valid salary ranges are available for this dataset.")
else:
    salary_groups = (
        salary_rows.groupby(["salary_currency", "salary_period"], dropna=False)
        .agg(
            postings=("salary_midpoint", "size"),
            median_midpoint=("salary_midpoint", "median"),
            median_minimum=("salary_min", "median"),
            median_maximum=("salary_max", "median"),
        )
        .sort_values("postings", ascending=False)
    )
    display(salary_groups)
    print("Compare salary values only within the same currency and pay period.")

## 8. Key takeaways and export

The following summary is calculated from the cleaned data, so it remains up to date if you replace the CSV with a new compatible extract. The final cell exports the cleaned dataset and embeds the dashboard data so `dashboard.html` works offline.

In [ ]:
top_state = df["system_state"].value_counts().head(1)
top_system = df["health_system"].value_counts().head(1)
top_family = df["job_family"].value_counts().head(1)
top_source = df["source"].value_counts().head(1)
salary_percent = df["salary_reported"].mean() * 100

print(f"• Most represented health-system state: {top_state.index[0]} ({top_state.iloc[0]:,} postings)")
print(f"• Most represented health system: {top_system.index[0]} ({top_system.iloc[0]:,} postings)")
print(f"• Most common inferred job family: {top_family.index[0]} ({top_family.iloc[0]:,} postings)")
print(f"• Most common posting source: {top_source.index[0]} ({top_source.iloc[0]:,} postings)")
print(f"• Listings with a usable salary range: {salary_percent:.1f}%")
print("\nInterpretation: these findings describe this sample and its collection window, not all US healthcare hiring.")

In [ ]:
cleaned_file = OUTPUT_DIR / "cleaned_hospital_jobs.csv"
df.to_csv(cleaned_file, index=False)
print(f"Saved cleaned dataset: {cleaned_file} ({len(df):,} rows)")

# Keep only fields needed for the offline, interactive dashboard.
dashboard_columns = {
    "state": "system_state",
    "system": "health_system",
    "family": "job_family",
    "employment": "employment_group",
    "source": "source",
    "posted": "posted_at",
    "salary_reported": "salary_reported",
}
dashboard_data = df[list(dashboard_columns.values())].rename(columns={
    source: target for target, source in dashboard_columns.items()
}).copy()
dashboard_data["posted"] = dashboard_data["posted"].dt.strftime("%Y-%m-%d")
dashboard_data = dashboard_data.astype(object).where(pd.notna(dashboard_data), None)
embedded_json = json.dumps(
    dashboard_data.to_dict(orient="records"),
    ensure_ascii=False,
    separators=(",", ":"),
).replace("<", "\\u003c")

dashboard_file = Path("dashboard.html")
if not dashboard_file.exists():
    raise FileNotFoundError(
        "dashboard.html was not found. Keep the dashboard file beside this notebook."
    )
html = dashboard_file.read_text(encoding="utf-8")
placeholder = "<script id=\"dashboard-data\" type=\"application/json\">__DATA__</script>"
if html.count(placeholder) != 1:
    raise ValueError("Expected exactly one dashboard data placeholder in dashboard.html.")
dashboard_file.write_text(
    html.replace(placeholder, f'<script id="dashboard-data" type="application/json">{embedded_json}</script>'),
    encoding="utf-8",
)
print(f"Updated {dashboard_file} with {len(dashboard_data):,} dashboard records.")
print(f"Dashboard file size: {dashboard_file.stat().st_size / 1_000_000:.2f} MB")